# Array

Inspect arrays, calculate with matrices and vectors, prepare data, and summarize observations.

Run the setup cell first. Each numbered section has editable inputs followed by a
calculation cell. Run that section again after changing its inputs, or use
**Restart Kernel and Run All Cells** to run the examples from a clean state.
Run each workflow's calculation before its result-detail cells. Every input has
a comment describing its format or choices. Optional fields accept `None`,
`"None"`, or blank text. Rerun setup after updating these files.

Matrices accept `1 2; 3 4`, rows on separate lines, fractions such as `1/3`,
nested lists, and NumPy arrays. Rows usually represent samples and columns features.
Results shown with `cache=cache` are saved automatically for use in any notebook
through `cache.load("name")`. The [Cache notebook](09_cache.ipynb) manages them.

[Setup and notebook index](README.md)

In [1]:
from pathlib import Path
import sys

# Works when Jupyter starts in this folder, the repository root, or a subfolder.
PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "CommonUtils.py").is_file() and (p / "notebooks" / "notebook_tools.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Start Jupyter from the ee2211-solver repository or its notebooks folder.")
for folder in (PROJECT_ROOT, PROJECT_ROOT / "notebooks"):
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))

import numpy as np
import pandas as pd
import sympy as sp
from IPython.display import Markdown, display
import importlib
import notebook_tools
importlib.reload(notebook_tools)
from notebook_tools import (NotebookCache, matrix, optional_matrix, is_blank,
    number, integer, boolean, variable_names, coordinates, choice, show, show_metrics)

pd.set_option("display.precision", 10)
cache = NotebookCache()
print(f"Python {sys.version.split()[0]} | shared cache: {cache.path}")

Python 3.11.17 | shared cache: D:\NUS\Y2S1\EE2211\solvers\ee2211-solver\notebooks\.cache\matrices.sqlite3


## 1. Inspect an array
The ordinary inverse needs a square full-rank matrix. A left inverse needs full
column rank; a right inverse needs full row rank. The transpose and pseudoinverse
are also displayed for reuse, including for singular matrices.

In [2]:
values = '1 2; 3 4'  # Numeric matrix: "1 2;3 4", nested rows, or an array.

In [3]:
from notebook_tools import inspect_matrix

array = matrix(values)
details, inverses = inspect_matrix(array)
show("array", array, cache=cache, source="Array inspection", kind="input")
show_metrics(details)
show("transpose", array.T, cache=cache, source="Array inspection")
show("pseudoinverse", np.linalg.pinv(array), cache=cache, source="Array inspection")
for name, value in inverses:
    show(name, value, cache=cache, source="Array inspection")

**array**

,0,1
0,1.0,2.0
1,3.0,4.0


,Value
shape,2 x 2
rank,2
determinant,-2
inverse,yes
left inverse,yes
right inverse,yes


**transpose**

,0,1
0,1.0,3.0
1,2.0,4.0


...

,0,1
0,-2.0,1.0
1,1.5,-0.5


**left_inverse**

,0,1
0,-2.0,1.0
1,1.5,-0.5


**right_inverse**

,0,1
0,-2.0,1.0
1,1.5,-0.5


## 2. Matrix and vector calculations
Choose `Dot product`, `Matrix product`, `Add`, `Subtract`, `Elementwise product`,
or `Trace`. Trace only uses A. Dot product accepts row or column vectors;
matrix product requires A's columns to match B's rows.

In [4]:
operation = 'Dot product'  # Dot product | Matrix product | Add | Subtract | Elementwise product | Trace.
A = '2; -1; 3'  # First numeric matrix or vector.
B = '0; 4; 1'  # Second matrix/vector; ignored for Trace.

In [5]:
from DataPreparation import matrix_calculation

calculation_result = matrix_calculation(matrix(A), operation, None if operation == "Trace" else matrix(B))
show("matrix_result", calculation_result, cache=cache, source=operation)

**matrix_result**

,0
0,-1.0


## 3. Prepare data
Choose `Min-max scaling`, `Z-score standardization`, or `Mean imputation`.
Statistics are fitted independently per feature from observed reference rows.
Z-scores use population standard deviation. Scaling preserves values outside the
reference range; it does not clip them.

Leave `reference`, `first`, and `second` as `None` to fit on the input.
For supplied min-max parameters, first = minimums and second = maximums.
For supplied z-score parameters, first = means and second = standard deviations.
Inverse scaling requires original reference rows or both supplied parameters.
Missing entries use `?` or `NA`, or your chosen numeric marker.

In [6]:
method = 'Min-max scaling'  # Min-max scaling | Z-score standardization | Mean imputation.
data = '10; 18; 30'  # Samples by feature; ? or NA marks missing entries.
reference = None  # Training/reference rows, or None to fit on data.
first = None  # Minimums (min-max) / means (z-score), or None to fit.
second = None  # Maximums / standard deviations, or None to fit.
output_range = (0, 1)  # (low, high), with low < high; min-max only.
inverse = False  # True reverses scaling; False applies it.
missing_marker = None  # Optional missing-value number, e.g. -1; otherwise None.

In [7]:
from DataPreparation import prepare_data

preparation = prepare_data(
    matrix(data, allow_missing=True), method,
    reference=optional_matrix(reference, allow_missing=True),
    first=optional_matrix(first),
    second=optional_matrix(second),
    output_range=output_range, inverse=boolean(inverse), missing_marker=number(missing_marker, optional=True),
)
show("prepared_data", preparation["result"], cache=cache, source=method)
parameters = {"First parameter": preparation["first"], "Missing inputs": preparation["missing"], "Observed reference rows": preparation["observed"]}
if preparation["second"] is not None:
    parameters["Second parameter"] = preparation["second"]
display(pd.DataFrame(parameters))

**prepared_data**

,0
0,0.0
1,0.4
2,1.0


,First parameter,Missing inputs,Observed reference rows,Second parameter
0,10.0,0,3,30.0


### Worked example: mean imputation
The missing value below is replaced by the observed mean, 6. You can change the
input or supply reference training rows to `prepare_data` in the previous section.

In [8]:
imputation_raw = "2;4;12;?"  # Numeric column; ? marks the missing entry.
imputation = prepare_data(matrix(imputation_raw, allow_missing=True), "Mean imputation")
show("imputed_data", imputation["result"], cache=cache, source="Mean imputation")

**imputed_data**

,0
0,2.0
1,4.0
2,12.0
3,6.0


## 4. Quartiles and descriptive statistics
Choose `Median of halves` or `Linear interpolation`. Median of halves excludes
the middle observation when the sample count is odd. Each feature is summarized
separately; fences are Q1 − 1.5 IQR and Q3 + 1.5 IQR.

In [9]:
observations = '1; 2; 3; 4; 8; 9; 10; 12'  # Numeric rows; summarize each feature column.
quartiles = 'Median of halves'  # Median of halves | Linear interpolation.

In [10]:
from DataPreparation import descriptive_statistics

statistics = descriptive_statistics(matrix(observations), quartiles)
statistic_names = ("minimum", "q1", "median", "q3", "maximum", "iqr", "mean", "population_std", "lower_fence", "upper_fence")
display(pd.DataFrame({key: statistics[key] for key in statistic_names}))
print(f"Observations per feature: {statistics['count']}")
quartile_summary = np.column_stack([statistics[key] for key in statistic_names[:6]])
cache.save("quartile_summary", quartile_summary, source="Minimum, Q1, median, Q3, maximum, IQR")

,minimum,q1,median,q3,maximum,iqr,mean,population_std,lower_fence,upper_fence
0,1.0,2.5,6.0,9.5,12.0,7.0,6.125,3.8547859863,-8.0,20.0


Observations per feature: 8


'8c096e35543a45778881904f74411413'